In [17]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import Faithfulness, AnswerRelevancy,ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
loader = PyPDFLoader("attention.pdf")
chunks = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 100)
docs = text_splitter.split_documents(chunks)
embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
db = FAISS.from_documents(docs, embeddings)
retriever = db.as_retriever(k=5)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_23236\2900079490.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [3]:
llm = ChatGroq(model="llama-3.1-8b-instant")

In [4]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using this context:\n{context}"),
    ("human", "{question}")
])

In [5]:
def ask(question):
    retrieved = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in retrieved)
    answer = llm.invoke(prompt.format_messages(context=context, question=question)).content
    return answer, retrieved

Evaluation

In [6]:
ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

In [8]:
faithfulness_metric = Faithfulness(llm=ragas_llm)
answer_relevancy_metric = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
context_recall_metric = ContextRecall(llm=ragas_llm)

In [9]:
metrics = [
        faithfulness_metric,
        answer_relevancy_metric,
        context_recall_metric
    ]

In [26]:
# Buffer to collect Q&A pairs for batch evaluation
eval_buffer = []

In [19]:
def run(question, reference=None):
    """
    Run a query through the RAG system and store it for evaluation.
    
    Args:
        question (str): user query
        reference (str, optional): ground truth answer
    """
    answer, retrieved = ask(question)

    buffer.append(SingleTurnSample(
        user_input=question,
        response=answer,
        retrieved_contexts=[d.page_content for d in retrieved],
        reference=reference  # optional
    ))

    return answer

In [27]:
def run(question, ground_truth=None):
    if not question.strip():          # guard against empty input
        print("Empty question, skipping.")
        return
    answer, retrieved = ask(question)
    eval_buffer.append(SingleTurnSample(
        user_input=question.strip(),  # strip whitespace
        response=answer,
        retrieved_contexts=[d.page_content for d in retrieved],
        reference=ground_truth
    ))
    print("Bot:", answer)

# --- Evaluate everything in buffer ---
def run_eval():
    if not eval_buffer:
        print("Nothing to evaluate."); return
    active_metrics = metrics if any(s.reference for s in eval_buffer) else metrics[:2]
    results = evaluate(EvaluationDataset(samples=eval_buffer), metrics=active_metrics).to_pandas()
    cols = ["user_input", "faithfulness", "answer_relevancy"] + \
           (["context_recall"] if any(s.reference for s in eval_buffer) else [])
    print(results[cols].to_string(index=False))
    eval_buffer.clear()

# --- Chat loop ---
print("Type 'eval' to evaluate, 'exit' to quit")
print("With ground truth: question | expected answer\n")
while True:
    q = input("You: ")
    if q.lower() == "exit": break
    if q.lower() == "eval": run_eval(); continue
    if " | " in q:
        question, gt = q.split(" | ", 1)
        run(question.strip(), gt.strip())
    else:
        run(q)
    print(f"[{len(eval_buffer)} buffered]\n")

Type 'eval' to evaluate, 'exit' to quit
With ground truth: question | expected answer

Bot: Attention is a fundamental concept in deep learning and natural language processing (NLP) that allows a model to focus on specific parts of the input data when processing it. It's a mechanism that helps the model to selectively weigh the importance of different input elements, such as words or tokens, when generating an output.

In the context of NLP, attention is often used in models like recurrent neural networks (RNNs) and transformers to help them understand the relationships between different words or tokens in a sentence. It's particularly useful when dealing with long-range dependencies, such as understanding the meaning of a sentence that spans multiple clauses or phrases.

There are two main types of attention:

1. **Self-attention**: This type of attention is used within a single sequence, such as a sentence, to compute a representation of the sequence. Self-attention allows the model 

Evaluating: 100%|██████████| 4/4 [02:56<00:00, 44.10s/it]


        user_input  faithfulness  answer_relevancy
what is attention?           NaN          0.294464
              eva;           0.0          0.000000
Empty question, skipping.
[0 buffered]

